# Week 10 · Day 2 — Comparing CNN Architectures on a Real Task (Scene Recognition)

**You're in the field now.** Today we classify **natural scenes** — buildings, forest, glacier, mountain, sea, street — and put the famous CNN architectures head-to-head to see which is best *for this job*.

- Dataset: **Intel Image Classification** — ~14k train / ~3k test color images, **6 scene classes**.
- We compare **5 architectures** from the CNN-history talk: **InceptionV3, ResNet50, DenseNet121, MobileNetV2, EfficientNetB0**.
- Method: **feature extraction** — freeze each pretrained backbone, train a small head.
- We compare on three real-world axes: **accuracy · speed · model size**.
- Then we take the winner and **fine-tune** it for a final boost.

> **Kaggle GPU:** Settings → Accelerator → GPU, then add the Intel Image Classification dataset via Add Input.

> **How this notebook is organised:** we train and evaluate **one architecture at a time**, recording its numbers, then compare them all at the end. Simple and sequential.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
import time

tf.random.set_seed(42)
print("TF version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

## 1. Load the data

- Intel Scenes is already split into `seg_train` and `seg_test`, each with one folder per class.
- On Kaggle the folders are **nested one level** (`seg_train/seg_train/<class>/`) — adjust the path if yours differs.
- `image_dataset_from_directory` reads class folders directly; each subfolder name becomes a label.

In [ ]:
# Kaggle path (note the nesting) — set to the folder that CONTAINS the class subfolders
TRAIN_DIR = "/kaggle/input/intel-image-classification/seg_train/seg_train"
TEST_DIR  = "/kaggle/input/intel-image-classification/seg_test/seg_test"

IMG_SIZE = (150, 150)   # Intel images are ~150x150
BATCH = 32
AUTOTUNE = tf.data.AUTOTUNE

train_ds = keras.utils.image_dataset_from_directory(
    TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH, label_mode="int", shuffle=True, seed=42)
test_ds = keras.utils.image_dataset_from_directory(
    TEST_DIR, image_size=IMG_SIZE, batch_size=BATCH, label_mode="int", shuffle=False)

class_names = train_ds.class_names
n_classes = len(class_names)
print("classes:", class_names)

# grab the test labels once (for metrics later)
y_test = np.concatenate([y.numpy() for _, y in test_ds])

train_ds = train_ds.prefetch(AUTOTUNE)
test_ds  = test_ds.prefetch(AUTOTUNE)

In [ ]:
# look at a few images per class
plt.figure(figsize=(14, 4))
for images, labels in train_ds.take(1):
    for i in range(12):
        plt.subplot(2, 6, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        plt.title(class_names[labels[i]], fontsize=9)
        plt.axis("off")
plt.suptitle("Intel scenes — 6 classes")
plt.tight_layout(); plt.show()

## 2. A helper to build, train & evaluate one model

- To keep the comparison fair, every architecture uses the **same recipe**: freeze the pretrained backbone, add `preprocess_input` + pooling + a Dense head, train only the head.
- One function does it end to end and returns the numbers we care about: **accuracy, macro-F1, training time, inference time, params.**
- We keep macro-F1 (a balanced-average score) as good practice, even though Intel's classes are fairly balanced.

In [ ]:
results = []   # each model appends its numbers here

def run_architecture(name, ctor, preprocess, epochs=4):
    """Build (frozen backbone + new head), train the head, evaluate. Returns the model."""
    # build
    base = ctor(weights="imagenet", include_top=False, input_shape=(150, 150, 3))
    base.trainable = False
    inputs = keras.Input((150, 150, 3))
    x = preprocess(inputs)
    x = base(x, training=False)
    x = keras.layers.GlobalAveragePooling2D()(x)
    x = keras.layers.Dropout(0.3)(x)
    outputs = keras.layers.Dense(n_classes, activation="softmax")(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer=keras.optimizers.Adam(1e-3),
                  loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    # train (head only)
    t0 = time.time()
    model.fit(train_ds, epochs=epochs, verbose=0)
    train_time = time.time() - t0

    # evaluate
    t1 = time.time()
    preds = model.predict(test_ds, verbose=0).argmax(1)
    infer_time = time.time() - t1
    acc = (preds == y_test).mean()
    macro_f1 = f1_score(y_test, preds, average="macro")
    params = model.count_params()

    results.append({"model": name, "accuracy": acc, "macro_f1": macro_f1,
                    "train_s": train_time, "infer_s": infer_time, "params": params})
    print(f"{name}:  acc {acc:.2%} | macro-F1 {macro_f1:.3f} | train {train_time:.0f}s | infer {infer_time:.1f}s | params {params/1e6:.1f}M")
    return model

## 3. Train & evaluate each architecture, one at a time

Each cell below runs one model. Watch the numbers appear — you'll start to see the trade-offs (big vs small, fast vs accurate) before we even tabulate them.

### 3a. InceptionV3  — *multi-scale filters (2014)*

In [ ]:
run_architecture("InceptionV3", keras.applications.InceptionV3,
                 keras.applications.inception_v3.preprocess_input);

### 3b. ResNet50  — *skip connections (2015)*

In [ ]:
run_architecture("ResNet50", keras.applications.ResNet50,
                 keras.applications.resnet50.preprocess_input);

### 3c. DenseNet121  — *dense feature reuse (2016)*

In [ ]:
run_architecture("DenseNet121", keras.applications.DenseNet121,
                 keras.applications.densenet.preprocess_input);

### 3d. MobileNetV2  — *lightweight, for phones (2017)*

In [ ]:
run_architecture("MobileNetV2", keras.applications.MobileNetV2,
                 keras.applications.mobilenet_v2.preprocess_input);

### 3e. EfficientNetB0  — *compound scaling (2019)*

In [ ]:
run_architecture("EfficientNetB0", keras.applications.EfficientNetB0,
                 keras.applications.efficientnet.preprocess_input);

## 4. The results table

In [ ]:
res = pd.DataFrame(results).sort_values("macro_f1", ascending=False).reset_index(drop=True)
d = res.copy()
d["accuracy"] = (d["accuracy"] * 100).round(1).astype(str) + "%"
d["macro_f1"] = d["macro_f1"].round(3)
d["train_s"] = d["train_s"].round(0).astype(int)
d["infer_s"] = d["infer_s"].round(1)
d["params"] = (d["params"] / 1e6).round(1).astype(str) + "M"
d.columns = ["Model", "Accuracy", "Macro-F1", "Train (s)", "Infer (s)", "Params"]
print(d.to_string(index=False))

## 5. Visualize the trade-offs

- **Accuracy per model** — who classifies scenes best.
- **Accuracy vs size** — the efficiency story from the CNN-history talk, made real.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

xr = np.arange(len(res))
ax1.bar(xr, res["accuracy"] * 100, color="steelblue")
ax1.set_xticks(xr); ax1.set_xticklabels(res["model"], rotation=30, ha="right")
ax1.set_ylabel("test accuracy (%)"); ax1.set_title("Accuracy by architecture"); ax1.grid(alpha=0.3)
for i, v in enumerate(res["accuracy"] * 100):
    ax1.text(i, v + 0.5, f"{v:.1f}", ha="center", fontsize=9)

ax2.scatter(res["params"] / 1e6, res["accuracy"] * 100, s=140, color="green")
for _, r in res.iterrows():
    ax2.annotate(r["model"], (r["params"] / 1e6, r["accuracy"] * 100),
                 textcoords="offset points", xytext=(6, 4), fontsize=9)
ax2.set_xlabel("parameters (millions)"); ax2.set_ylabel("test accuracy (%)")
ax2.set_title("Accuracy vs model size"); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**How to read this (the field skill):**
- The most *accurate* model isn't always the right choice. A phone app wants **MobileNet** (tiny, fast); a server that just needs the best score might take the heavier net.
- Look at **accuracy per million parameters** — the small models (MobileNet, EfficientNet) are often astonishingly efficient.
- This trade-off *is* the job: there's no single "best" — only best-for-a-purpose.

## 6. Fine-tune the winner

- Take the top model by macro-F1.
- **Unfreeze** the backbone and train a little more with a **very small learning rate** (100× smaller) — gently adapting ImageNet features to scenes.
- This squeezes out the final accuracy.

In [ ]:
winner = res.iloc[0]["model"]
print("winner by macro-F1:", winner)

ARCHS = {
    "InceptionV3":    (keras.applications.InceptionV3,    keras.applications.inception_v3.preprocess_input),
    "ResNet50":       (keras.applications.ResNet50,       keras.applications.resnet50.preprocess_input),
    "DenseNet121":    (keras.applications.DenseNet121,    keras.applications.densenet.preprocess_input),
    "MobileNetV2":    (keras.applications.MobileNetV2,    keras.applications.mobilenet_v2.preprocess_input),
    "EfficientNetB0": (keras.applications.EfficientNetB0, keras.applications.efficientnet.preprocess_input),
}
ctor, prep = ARCHS[winner]

# rebuild + train the head
base = ctor(weights="imagenet", include_top=False, input_shape=(150, 150, 3))
base.trainable = False
inputs = keras.Input((150, 150, 3))
x = prep(inputs); x = base(x, training=False)
x = keras.layers.GlobalAveragePooling2D()(x); x = keras.layers.Dropout(0.3)(x)
outputs = keras.layers.Dense(n_classes, activation="softmax")(x)
model = keras.Model(inputs, outputs)
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(train_ds, epochs=4, verbose=0)

# fine-tune: unfreeze, tiny LR
base.trainable = True
model.compile(optimizer=keras.optimizers.Adam(1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(train_ds, epochs=3, verbose=1)

preds = model.predict(test_ds, verbose=0).argmax(1)
print(f"\n{winner} after fine-tuning:  acc {(preds==y_test).mean():.2%} | macro-F1 {f1_score(y_test,preds,average='macro'):.3f}")

In [ ]:
cm = confusion_matrix(y_test, preds)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap="Blues", colorbar=False, xticks_rotation=45)
plt.title(f"{winner} (fine-tuned) — scene classification")
plt.tight_layout(); plt.show()

print(classification_report(y_test, preds, target_names=class_names, digits=3))

- The confusion matrix shows which scenes get mixed up — often the visually similar pairs (glacier ↔ mountain, sea ↔ glacier), the same ones a person might hesitate on.
- Per-class recall tells you where the model is weak, class by class.

## Your turn (solo task) ✍️

Pick at least two:
1. **Fine-tune a different model** from the table and compare it to the winner.
2. **Train the top two longer** (more epochs) — does the ranking change?
3. **Add data augmentation** (random flip/rotation) before the backbone — does accuracy improve?
4. **Predict on `seg_pred`** (the unlabeled folder) with your best model and eyeball a few predictions.

In [ ]:
# ===== YOUR EXPERIMENTS HERE =====



## Where CNNs are used in the real world

Scene recognition is one small corner of what convolutional networks do in production. The same "extract visual features → decide" machinery powers a huge range of applications:

### 🖼️ Image classification & organization
- **Photo apps** — auto-tagging and grouping photos ("beaches", "mountains", "documents"), the direct cousin of today's scene task.
- **Content moderation** — flagging unsafe or unwanted images at scale.
- **Visual search** — "find products that look like this."

### 🗺️ Location & mapping
- **Geo-tagging** — inferring where a photo was taken from its scene content.
- **Satellite & aerial imagery** — land-use mapping, deforestation and crop monitoring, disaster response.

### 🚗 Autonomous systems
- **Self-driving cars & drones** — recognizing road, sky, obstacles, and lanes for navigation.
- **Robotics** — letting a robot understand and move through its surroundings.

### 🏥 Medicine & science
- **Medical imaging** — detecting disease in X-rays, CT scans, dermatoscopy, and pathology slides.
- **Microscopy & astronomy** — classifying cells, particles, and galaxies.

### 🏭 Industry & security
- **Manufacturing** — spotting defects on production lines (visual quality control).
- **Agriculture** — identifying crop disease and weeds from leaf images.
- **Security & surveillance** — detection and recognition in camera feeds.

> Every one of these starts the same way you did today: a pretrained CNN backbone that already "sees," adapted with a small amount of task-specific training. **That is why transfer learning is the workhorse of applied computer vision.**

## Summary

- **Real task, clean data:** classifying 6 natural-scene types from the Intel dataset.
- **Architecture comparison, one at a time:** InceptionV3, ResNet50, DenseNet121, MobileNetV2, EfficientNetB0 — each trained and scored, then tabulated.
- **Three axes that matter in the field:** accuracy, speed, and model size. No single winner — MobileNet is tiny and fast; heavier nets may score a little higher.
- **Fine-tuning the winner** (unfreeze + tiny LR) gave a final boost.
- The real skill isn't "pick the best model" — it's picking the **right model for the constraints** (device, latency, accuracy needs).

**Tomorrow:** object detection — not just *what* is in an image, but *where*.

---
*A PyTorch version of this notebook is provided separately — same comparison, same dataset, using `torchvision.models`.*